# 一、消息队列
## 举例1：JSON格式

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model = init_chat_model(
model="gpt-5.4-mini",
model_provider="openai",
api_key=CLOSEAI_API_KEY,
base_url=CLOSEAI_BASE_URL
)

In [ ]:
# 通过JSON初始化
messages = [
    {"role": "system", "content": "你是一个善于给出通俗易懂解释的AI助手"},
    {"role": "user", "content": "你好"},
    {"role": "assistant", "content": "你好！我能帮你什么？"},
    {"role": "user", "content": "什么是机器学习"}
]
response = model.invoke(messages)
print(response.content)

## 举例2：消息对象列表

In [ ]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

# 从.env文件中加载环境变量
load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")
model = init_chat_model(
model="gpt-5.4-mini",
model_provider="openai",
api_key=CLOSEAI_API_KEY,
base_url=CLOSEAI_BASE_URL
)
# 通过JSON初始化（重点在这里）
messages = [
    SystemMessage("你是一个善于给出通俗易懂解释的AI助手"),
    HumanMessage("你好"),
    AIMessage("你好！我能帮你什么？"),
    HumanMessage("什么是机器学习"),
]

response = model.invoke(messages)
print(response.content)

# 二、HumanMessage的使用
#举例1 通过CloseAI平台调用 gpt-5.4-mini 展示 name 的作用

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
CLOSEAI_API_KEY = os.getenv("CLOSEAI_API_KEY")
CLOSEAI_BASE_URL = os.getenv("CLOSEAI_BASE_URL")

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=CLOSEAI_API_KEY,
    base_url=CLOSEAI_BASE_URL
)

messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息 可能带有 name 字段。你的任务是：严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出 格式：{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        additional_kwargs={"name": "Bob"}
    ),
    HumanMessage(
        content="我认为 1+1>2",
        additional_kwargs={"name": "Tom"}
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        additional_kwargs={"name": "audience"}
    )
]

response = model.invoke(messages)
print(response.content)

In [ ]:
"""
#把`name=xxx`挪进`additional_kwargs={...}`里面
修改前（有红波浪）
HumanMessage(
    content="我认为 1+1=2",
    name="Bob"
)
修改后（消除红波浪）
HumanMessage(
    content="我认为 1+1=2",
    additional_kwargs={"name": "Bob"}
)

快递盒子（HumanMessage）

- `content` = 盒子里面的主物品（必填）
- `additional_kwargs` = 盒子外面贴的小贴纸，用来写备注信息（自定义内容放这里）
你不能直接在盒子外壳乱刻字（直接写 name="Bob"），要写贴纸上面（放到 additional_kwargs 字典）
"""

# 三、AIMessage的使用

# 四、ToolMessage的使用
## 1.json格式
## 2.对象格式

# 五、对话历史优化

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话
    max_pairs: 保留的对话轮数（每轮 = user + assistant）
    """
    # 分离 system 和对话
    system_msgs = [m for m in messages if m.get("role") == "system"]
    conversation_msgs = [m for m in messages if m.get("role") != "system"]
    # 只保留最近的
    recent_msgs = conversation_msgs[-(max_pairs * 2):]
    # 返回：system + 最近对话
    return system_msgs + recent_msgs

In [ ]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第1轮
long_conversation.append(HumanMessage(content="什么是列表？用一句解释"))
r1 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r1.content))

# 第2轮
long_conversation.append(HumanMessage(content="列表和元组有什么区别？用一句解释"))
r2 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r2.content))

# 第3轮
long_conversation.append(HumanMessage(content="什么是字典呢？用一句解释"))
r3 = model.invoke(long_conversation)
long_conversation.append(AIMessage(content=r3.content))

print(f"原始消息数: {len(long_conversation)}")
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")

optimized.append(HumanMessage(content="我第一个问题问的是什么？"))
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

# 六、多轮聊天机器人

In [ ]:
import os
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek

# 将env文件中的变量加载为环境变量
# override=True：表示.env优先
load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    api_key=DEEPSEEK_API_KEY,
    api_base=DEEPSEEK_BASE_URL,
    model_name="deepseek-v4-flash"
)

# 1. 基础配置
MODEL_NAME = "deepseek-v4-flash"
MAX_PAIRS_HISTORY = 10 #滑动窗口最大保留轮数，最多保留10 轮完整对话
EXIT_WORD = "quit"

print(model.invoke("你好"))
#连通性测试，调用一次模型，验证 api_key、base_url 网络调用是否正常；能输出回复代表模型对象正常可用

# 3. 初始化消息列表
messages = [
    {
        "role":"system",
        "content":"你是小谷姐姐，尚硅谷教育的数字员工，也是一名耐心、友好的智能助手。我会用自然、清晰的方式回答用户问题。"
    }
]
# messages是整个聊天机器人的全部对话内存，使用 OpenAI 标准字典格式消息，整个列表保存全部聊天记录，不会被裁剪删除
# role:system：系统提示词，设定 AI 身份、性格、行为规则
# keep_recent_messages 函数会强制保留这条消息，永远不会被删掉，一旦丢失 system，AI 就忘记自己是谁

# 4. 启动提示
print(f"✨ 请输入问题，输入 {EXIT_WORD} 结束对话\n")

# 5. 多轮对话主循环
# 轮次记录
i = 1
while True:
    print("\n", "=" * 10, f'-> 第 {i} 轮对话开始 <-', "=" * 10, "\n")
    user_input = input("🙋 请输入：")

    # 退出判断
    if user_input.lower() == EXIT_WORD:
        print("🌙 对话已结束，欢迎下次再来！")
        break

    # 追加用户消息
    messages.append({"role":"user","content":user_input})
# 高频重点：拿到用户输入，立刻追加到全局完整消息列表messages
# 此时messages内部 = system 消息 + 历史全部对话 + 当前用户最新提问

    # 流式输出模型回复
    print("🧚 小谷姐姐：", end="", flush=True)
    reply_content = ""

    # 优化历史记忆
    memory_messages = keep_recent_messages(messages,max_pairs = MAX_PAIRS_HISTORY) # 控制发送给模型的消息长度

    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            reply_content += chunk.content

    print("\n", "=" * 10, f'-> 第 {i} 轮对话结束 <-', "=" * 10, "\n")
    i += 1
"""stream 流式迭代获取模型返回
1、model.stream()：LangChain 流式调用接口，对比model.invoke()：
    invoke()：等待模型全部生成完毕，一次性返回完整结果，用户需要长时间等待。
    stream()：模型一边生成一边返回碎片对象chunk，实现打字机实时输出，用户体验好。
2、for chunk in model.stream(memory_messages)：循环迭代每一个返回分片。
3、if chunk.content：部分分片没有文本内容，过滤空分片，避免打印空字符。
4、两件事同步做：
    打印分片到控制台，实时展示给用户。
    把分片字符串拼接进reply_content变量，拿到完整 AI 回答
"""

    # 追加 AI 回复
    messages.append({"role":"assistant","content":reply_content})

    #这里的代码是可以运行的
